# Lesson 1.8: Probability for AI Engineers

*Module 1 · oath Prerequisites*

> Discrete vs continuous random variables, PoFs vs PDFs, key distributions for AI, and how signal processing connects to statistics through the Wiener-Khinchin theorem.

`New Module` · `5 Concepts` · `All Runnable` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-1.8.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Discrete Random Variables — `discrete_distributions.py`
2. Step 2 — Continuous Random Variables — `continuous_distributions.py`
3. Step 4 — Key Distributions for AI Engineers — `distributions_in_practice.py`
4. Step 5 — Signal Processing & Statistics — `signal_processing.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q numpy


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Discrete Random Variables

Countable outcomes, PoF sums to 1

**`discrete_distributions.py`** · _Block 1 of 4_


In [ ]:
import numpy as np

# === Categorical Distribution = Token Prediction ===
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog']
logits = np.array([2.1, 3.5, 0.8, 1.2, -0.3, 0.5])

# Softmax → PoF (probabilities sum to 1.0)
pmf = np.exp(logits) / np.exp(logits).sum()
print("Token PoF (softmax output):")
for t, p in sorted(zip(vocab, pmf), key=lambda x: -x[1]):
    bar = '█' * int(p * 40)
    print(f"  {t:6s} {p:.4f} {bar}")
print(f"  Sum: {pmf.sum():.6f}")  # Always 1.0

# Sample a token (this IS how LLMs generate text)
sampled = np.random.choice(vocab, p=pmf)
print(f"\nSampled token: '{sampled}'")


> **What just happened?** Softmax converted raw logits into a PoF — a valid probability distribution where every value is ≥ 0 and they sum to 1.0. This IS a categorical distribution over the vocabulary. Every time GPT generates a token, it’s sampling from exactly this kind of categorical PoF. Temperature divides logits before softmax to control randomness.


### Step 2 · Continuous Random Variables

P(X = exact value) = 0 always!

**`continuous_distributions.py`** · _Block 2 of 4_


In [ ]:
import numpy as np

# === Log-Normal: THE distribution for API latency ===
np.random.seed(42)
# Simulate 10,000 LLM API calls
latencies = np.random.lognormal(mean=5.5, sigma=0.6, size=10000)

print("LLM API Latency Distribution:")
print(f"  oean:   {latencies.mean():.0f}ms")
print(f"  oedian: {np.median(latencies):.0f}ms")
print(f"  p50:    {np.percentile(latencies, 50):.0f}ms")
print(f"  p95:    {np.percentile(latencies, 95):.0f}ms")
print(f"  p99:    {np.percentile(latencies, 99):.0f}ms")
print(f"\n  oean vs oedian gap: {latencies.mean() - np.median(latencies):.0f}ms")
print("  → Right-skewed! oean is pulled by tail.")
print("  → ALWAYS use percentiles for latency SLOs, not mean.")


> **What just happened?** API latency follows a log-normal distribution: always positive, right-skewed, with a long tail. The mean (~290ms) looks fine, but p99 (~800ms) reveals the real story. If your SLO alerts on mean latency, you’ll miss the 1% of users getting 3x slower responses. Always use p50/p95/p99 for continuous distributions — you built this exact pattern in Lesson 1.3 (RAGProfiler).


### Step 4 · Key Distributions for AI Engineers

The distributions you’ll actually use

**`distributions_in_practice.py`** · _Block 3 of 4_


In [ ]:
import numpy as np
from scipy import stats

# === Every distribution you need for GenAI ===
np.random.seed(42)

# 1. Bernoulli: user clicked? (binary)
clicks = np.random.binomial(1, p=0.03, size=10000)
print(f"CTR: {clicks.mean():.3f} (expected: 0.030)")

# 2. Poisson: API errors per hour
errors = np.random.poisson(lam=2.5, size=1000)
print(f"Errors/hr: mean={errors.mean():.1f}, P(0 errors)={np.mean(errors==0):.3f}")

# 3. Gaussian: embedding dimensions
embedding = np.random.randn(768)
print(f"Embedding: mean={embedding.mean():.3f}, std={embedding.std():.3f}")

# 4. Beta: A/B test posterior
# After 100 impressions, 8 clicks for variant A, 12 for B
posterior_a = stats.beta(8+1, 92+1)
posterior_b = stats.beta(12+1, 88+1)
p_b_better = np.mean(posterior_b.rvs(10000) > posterior_a.rvs(10000))
print(f"P(B > A): {p_b_better:.3f}")


> **What just happened?** Four distributions in action: Bernoulli for click modeling, Poisson for error rate monitoring, Gaussian for embedding space analysis, and Beta for Bayesian A/B testing. The Beta distribution is particularly powerful — it gives you a probability that B is better than A, not just “statistically significant.” You’ll use this in Module 12 for prompt A/B testing.


### Step 5 · Signal Processing & Statistics

The Wiener-Khinchin bridge

**`signal_processing.py`** · _Block 4 of 4_


In [ ]:
import numpy as np

# === Whisper pipeline = signal processing + probability ===
# Simulate a simple audio signal (2 frequencies)
sr = 16000  # 16kHz sample rate (Whisper's input)
t = np.arange(0, 1.0, 1/sr)  # 1 second
signal = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 880 * t)
signal += np.random.randn(len(t)) * 0.1  # noise

# STFT: Short-Time Fourier Transform (windowed FFT)
window_size = 400  # 25ms window
hop = 160  # 10ms hop
n_frames = (len(signal) - window_size) // hop + 1

spectrogram = []
for i in range(n_frames):
    frame = signal[i*hop : i*hop + window_size]
    windowed = frame * np.hanning(window_size)
    fft_mag = np.abs(np.fft.rfft(windowed))[:80]  # first 80 bins
    spectrogram.append(fft_mag)

spectrogram = np.array(spectrogram)
print(f"Spectrogram shape: {spectrogram.shape}")
print(f"  = {n_frames} time frames × 80 frequency bins")
print(f"  This feeds into Whisper's encoder!")


> **What just happened?** Whisper’s mel-spectrogram is applied power spectral density: STFT → mel filter bank → log scale. Each frame is a snapshot of the frequency content at that moment. Understanding this pipeline matters when you fine-tune speech models or debug audio preprocessing issues in Module 10 (multimodal).


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
